<a href="https://colab.research.google.com/github/AlbertPuentes/App_Deep-Learning-scritp_2/blob/main/Scrit2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

"""Métricas multivariadas y comparación de enfoques.
M1 Distancia de Mahalanobis | M2 PCA: SPE(Q) y Hotelling T2 | M3 Error AE por variable
M4 Concordancia entre métodos (Jaccard/Kappa) | M5 Sensibilidad del umbral."""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SENS = ['Diametro_mm','Peso_g','Presion_bar','Temperatura_C','Dureza_HRC','Tiempo_ciclo_s']
try:
    ent = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Entrenamiento')
    lot = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Lote a Inspeccionar')
except Exception:
    ent = pd.read_csv('CasoEstudioValvulas_Entrenamiento.csv'); lot = pd.read_csv('CasoEstudioValvulas_LoteAInspeccionar.csv')
ae = pd.read_csv('resultados_autoencoder.csv')          # del Script 1
Xtr, Xlot = ent[SENS].values, lot[SENS].values
sc = StandardScaler(); Ztr, Zlot = sc.fit_transform(Xtr), sc.transform(Xlot)

# ===== M1: MAHALANOBIS (usa la covarianza: detecta rupturas de correlación) =====
mu = Ztr.mean(0); S = np.cov(Ztr, rowvar=False); Sinv = np.linalg.pinv(S)
d2_lot = np.einsum('ij,jk,ik->i', Zlot-mu, Sinv, Zlot-mu)
d2_tr  = np.einsum('ij,jk,ik->i', Ztr-mu,  Sinv, Ztr-mu)
u_maha = st.chi2.ppf(0.99, df=6)
m_maha = d2_lot > u_maha

# ===== M2: PCA -> SPE (Q, residual) y T2 (espacio del modelo) =====
p = PCA(n_components=2).fit(Ztr)                 # 2 CP explican >90% de la varianza
Tlot = p.transform(Zlot); Ttr = p.transform(Ztr)
SPE_lot = ((Zlot - p.inverse_transform(Tlot))**2).sum(1)
SPE_tr  = ((Ztr  - p.inverse_transform(Ttr))**2).sum(1)
u_spe = np.percentile(SPE_tr, 99)
lam = p.explained_variance_
T2_lot = ((Tlot**2)/lam).sum(1); T2_tr = ((Ttr**2)/lam).sum(1)
u_t2 = np.percentile(T2_tr, 99)
m_spe, m_t2 = SPE_lot > u_spe, T2_lot > u_t2

# ===== M3: error AE por variable (diagnóstico) =====
err = ae[['ErrVar_'+c for c in SENS]].values
sensor_dom = [SENS[i] for i in err.argmax(1)]

# ===== M4: concordancia =====
m_ae = ae['Anomalia_AE'].values.astype(bool); m_uni = ae['Fuera_de_rango_UNI'].values.astype(bool)
def jac(a, b): return (a & b).sum() / max((a | b).sum(), 1)
def kappa(a, b):
    po = (a == b).mean(); pe = a.mean()*b.mean() + (1-a.mean())*(1-b.mean()); return (po-pe)/(1-pe)

# ===== M5: sensibilidad del umbral AE =====
mse_lot = ae['MSE_reconstruccion'].values
mse_tr_ae = None
sens = {f'P{q}': (mse_lot > np.percentile(mse_lot[m_ae], 0)).sum() for q in [95]}  # placeholder
tabla = pd.DataFrame({'ID': lot['ID_Valvula'], 'Mahalanobis': m_maha, 'SPE_PCA': m_spe,
                      'T2_PCA': m_t2, 'Autoencoder': m_ae, 'Univariado_3s': m_uni,
                      'Sensor_dominante': sensor_dom})
tabla['Votos_multivariados'] = tabla[['Mahalanobis','SPE_PCA','Autoencoder']].sum(1)
final = tabla['Votos_multivariados'] >= 2
print(tabla.groupby('Votos_multivariados')['ID'].apply(list))
print('Jaccard(AE, Maha)=%.2f  Jaccard(AE, SPE)=%.2f  Jaccard(AE, UNI)=%.2f'
      % (jac(m_ae, m_maha), jac(m_ae, m_spe), jac(m_ae, m_uni)))
print('Kappa(AE, UNI)=%.2f  -> acuerdo pobre: los métodos ven cosas DISTINTAS' % kappa(m_ae, m_uni))
print('FINAL (consenso >=2 metodos):', tabla.loc[final, 'ID'].tolist())

# ===== Evaluación tomando el consenso como referencia =====
tp = (m_uni & final).sum(); fp = (m_uni & ~final).sum(); fn = (~m_uni & final).sum()
print(f'Univariado vs consenso: Precision={tp/(tp+fp):.3f} Recall={tp/(tp+fn):.3f} '
      f'F1={2*tp/(2*tp+fp+fn):.3f}')
tp2 = (m_ae & final).sum(); fp2 = (m_ae & ~final).sum(); fn2 = (~m_ae & final).sum()
print(f'Autoencoder vs consenso: Precision={tp2/(tp2+fp2):.3f} Recall={tp2/(tp2+fn2):.3f}')

# ===== Gráficas =====
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ax[0].scatter(range(100), d2_lot, c=m_maha, cmap='coolwarm'); ax[0].axhline(u_maha, color='r', ls='--')
ax[0].set_title(f'M1 Mahalanobis (χ²99%,6={u_maha:.1f})'); ax[0].grid(alpha=.3)
ax[1].scatter(range(100), SPE_lot, c=m_spe, cmap='coolwarm'); ax[1].axhline(u_spe, color='r', ls='--')
ax[1].set_title('M2 SPE/Q de PCA (umbral P99)'); ax[1].grid(alpha=.3)
Zp = PCA(n_components=2).fit_transform(Zlot)
ax[2].scatter(Zp[:,0], Zp[:,1], c=final, cmap='coolwarm', s=45, edgecolors='k')
ax[2].set_title('M2bis PC1 vs PC2 (rojo=anómala)'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('figuras_script2.png', dpi=200); plt.close()
sns.heatmap(ent[SENS].corr(), annot=True, cmap='coolwarm', fmt='.2f',
            cbar_kws={'shrink': .8}).set_title('Fig.7 Correlación entre sensores (entrenamiento)')
plt.tight_layout(); plt.savefig('figuras_script2_corr.png', dpi=200); plt.close()
tabla.to_csv('resultados_metricas.csv', index=False)
print('Gráficas: figuras_script2.png, figuras_script2_corr.png | tabla: resultados_metricas.csv')


Votos_multivariados
0    [L061, L039, L018, L069, L030, L010, L005, L00...
1                             [L077, L083, L076, L075]
2                                   [L096, L067, L072]
3    [L071, L089, L056, L074, L085, L073, L078, L09...
Name: ID, dtype: object
Jaccard(AE, Maha)=0.81  Jaccard(AE, SPE)=0.96  Jaccard(AE, UNI)=0.31
Kappa(AE, UNI)=0.33  -> acuerdo pobre: los métodos ven cosas DISTINTAS
FINAL (consenso >=2 metodos): ['L071', 'L089', 'L056', 'L074', 'L085', 'L073', 'L078', 'L095', 'L090', 'L079', 'L098', 'L003', 'L087', 'L100', 'L086', 'L096', 'L082', 'L080', 'L097', 'L081', 'L093', 'L067', 'L084', 'L094', 'L072', 'L091', 'L099', 'L092']
Univariado vs consenso: Precision=0.611 Recall=0.393 F1=0.478
Autoencoder vs consenso: Precision=1.000 Recall=1.000
Gráficas: figuras_script2.png, figuras_script2_corr.png | tabla: resultados_metricas.csv
